In [ ]:
"""
@author: Chioma Opara and ClaudeAI
Last modified: Fri Feb 8 2026

Trains the DiT-XL/2 model from pretrained weights on a custom subset
of ImageNet-21K
"""

### Setting up environment

In [ ]:
# Clone the fast DiT repo
!git clone https://github.com/chuanyangjin/fast-DiT.git
%cd fast-DiT

# Install dependencies
!pip install diffusers timm accelerate --upgrade

Cloning into 'fast-DiT'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 103 (delta 42), reused 31 (delta 31), pack-reused 49 (from 2)
Receiving objects: 100% (103/103), 6.38 MiB | 14.17 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/fast-DiT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 46.7 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1


In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load Hugging Face token directly from environment variables for security
from google.colab import userdata
huggingface_token = userdata.get('HF_TOKEN')

# Hugging Face hub login
from huggingface_hub import login
login(token=huggingface_token)

### Downloading/Loading Pretrained Weights

In [ ]:
# Download pretrained weights
# !wget https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt -P /content/drive/MyDrive/ImageNet_Project/

--2026-05-04 21:24:20--  https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.62, 65.9.168.81, 65.9.168.4, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2700611775 (2.5G) [binary/octet-stream]
Saving to: ‘/content/drive/MyDrive/ImageNet_Project/DiT-XL-2-256x256.pt.3’

DiT-XL-2-256x256.pt 100%[===================>]   2.51G  80.7MB/s    in 31s     

2026-05-04 21:24:51 (83.9 MB/s) - ‘/content/drive/MyDrive/ImageNet_Project/DiT-XL-2-256x256.pt.3’ saved [2700611775/2700611775]



In [ ]:
# (If already downloaded) Copy pretrained weights to local storage
!cp -r /content/drive/MyDrive/ImageNet_Project/DiT-XL-2-256x256.pt\
      /content/DiT-XL-2-256x256.pt

### Feature Extraction

In [ ]:
# Copy dataset to Colab local storage first to make feature extraction faster
!cp -r /content/drive/MyDrive/ImageNet_Project/imagenet_subset_train \
      /content/imagenet_subset_train

In [ ]:
# Extract features from train images in drive
!torchrun --nnodes=1 --nproc_per_node=1 extract_features.py \
  --model DiT-XL/2 \
  --data-path /content/imagenet_subset_train/imagenet_subset_train \
  --features-path /content/drive/MyDrive/ImageNet_Project/features \
  --global-batch-size=256

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Starting rank=0, seed=0, world_size=1.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Rank 0:   0% 0/424 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
Rank 0: 100% 424/424 [1:44:26<00:00, 14.78s/it]


### Reading in extracted features

In [ ]:
# Read in zipped features files [Acknowledgment: AI generated code]
!mkdir -p /content/tmp_unzip
!mkdir -p /content/features/imagenet256_features
!mkdir -p /content/features/imagenet256_labels

import glob, os, shutil

zip_dir = "/content/drive/MyDrive/ImageNet_Project"
zips = sorted(glob.glob(f"{zip_dir}/*.zip"))

for z in zips:
    print(f"\nProcessing {os.path.basename(z)}")

    # clear temp folder
    shutil.rmtree("/content/tmp_unzip", ignore_errors=True)
    os.makedirs("/content/tmp_unzip", exist_ok=True)

    # unzip
    !unzip -q "{z}" -d /content/tmp_unzip

    if "features" in z:
        src = "/content/tmp_unzip/imagenet256_features"
        dst = "/content/features/imagenet256_features"
    elif "labels" in z:
        src = "/content/tmp_unzip/imagenet256_labels"
        dst = "/content/features/imagenet256_labels"
    else:
        continue

    if not os.path.exists(src):
        print(f"  ERROR: expected folder not found: {src}")
        continue

    files = os.listdir(src)
    print(f"  copying {len(files)} files → {dst}")

    for f in files:
        shutil.copy2(os.path.join(src, f), dst)

    print(f"  total now: {len(os.listdir(dst))}")

# clear temp folder
shutil.rmtree("/content/tmp_unzip", ignore_errors=True)


Processing imagenet256_features-001.zip
  copying 65535 files → /content/features/imagenet256_features
  total now: 65535

Processing imagenet256_features-002.zip
  copying 42997 files → /content/features/imagenet256_features
  total now: 108532

Processing imagenet256_labels-001.zip
  copying 65535 files → /content/features/imagenet256_labels
  total now: 65535

Processing imagenet256_labels-002.zip
  copying 42997 files → /content/features/imagenet256_labels
  total now: 108532


In [ ]:
# verify number of label and feature files
# !ls /content/features/imagenet256_features/ | wc -l
# !ls /content/features/imagenet256_labels/ | wc -l

108532
108532


###  Changes to be made to train.py script to allow initialization of an embedding table with the new 102 classes of images from my new dataset

In [ ]:
with open('/content/fast-DiT/train.py', 'r') as f:
    content = f.read()

# Fix 1 — ensure the CustomDataset class correctly encodes labels from 0-1101
# Fix 2 — load pretrained weights with shape mismatch handling
# Fix 3 — add resume loading logic
# Fix 4 — update checkpoint save to include train steps & fix model.module reference (to handle single GPU)
# Fix 5 — add time limit to training
# Fix 6 - Fix logger configuration for Colab/Jupyter
# Fix 7 — argparse: add new argparse arguments to load pretrained models and set default classes to 1102

# Fix 1 — ensure the customDataset class correctly encodes labels from 0-1101
old = """        return torch.from_numpy(features), torch.from_numpy(labels)"""

new = """        return torch.from_numpy(features), torch.from_numpy(labels)+1000"""

content = content.replace(old, new)

# Fix 2 & 3 — load pretrained weights with shape mismatch handling and add resume logic
old = """    # Prepare models for training:
    update_ema(ema, model, decay=0)  # Ensure EMA is initialized with synced weights
    model.train()  # important! This enables embedding dropout for classifier-free guidance
    ema.eval()  # EMA model should always be in eval mode
    model, opt, loader = accelerator.prepare(model, opt, loader)

    # Variables for monitoring/logging purposes:
    train_steps = 0
    log_steps = 0
    running_loss = 0
    start_time = time()"""

new = """    # Prepare models for training:
    update_ema(ema, model, decay=0)  # Ensure EMA is initialized with synced weights
    model.train()  # important! This enables embedding dropout for classifier-free guidance
    ema.eval()  # EMA model should always be in eval mode
    model, opt, loader = accelerator.prepare(model, opt, loader)

    # Load pretrained weights after accelerator.prepare()
    if args.pretrained:
        pretrained_checkpoint = torch.load(args.pretrained, map_location="cpu")
        pretrained_state = pretrained_checkpoint["ema"] if "ema" in pretrained_checkpoint else pretrained_checkpoint
        model_state = accelerator.unwrap_model(model).state_dict()

        loaded = 0
        skipped = []

        for k, v in pretrained_state.items():
            # SPECIAL CASE: class embedding table
            if k == "y_embedder.embedding_table.weight":
                old_rows = v.shape[0]
                model_state[k][:old_rows] = v
                loaded+=1
                continue

            # default logic
            if k in model_state and v.shape == model_state[k].shape:
                model_state[k] = v
                loaded+=1
            else:
              skipped.append(k)

        accelerator.unwrap_model(model).load_state_dict(model_state)
        if accelerator.is_main_process:
            logger.info(f"Loaded {loaded} pretrained weights; skipped {len(skipped)} mismatched tensors.")

    # Variables for monitoring/logging purposes:
    train_steps = 0
    log_steps = 0
    running_loss = 0
    start_time = time()
    # include timer for total training run
    run_start_time = time()

    if args.resume:
        resume_checkpoint = torch.load(args.resume, map_location="cpu")
        accelerator.unwrap_model(model).load_state_dict(resume_checkpoint["model"])
        ema.load_state_dict(resume_checkpoint["ema"])
        opt.load_state_dict(resume_checkpoint["opt"])
        train_steps = resume_checkpoint["train_steps"]
        if accelerator.is_main_process:
            logger.info(f"Resumed from checkpoint at step {train_steps}")"""

content = content.replace(old, new)

# Fix 4 — update checkpoint saving to include train steps
old = """                    checkpoint = {
                        "model": model.module.state_dict(),
                        "ema": ema.state_dict(),
                        "opt": opt.state_dict(),
                        "args": args
                    }"""

new = """                    checkpoint = {
                        "model": accelerator.unwrap_model(model).state_dict(),
                        "ema": ema.state_dict(),
                        "opt": opt.state_dict(),
                        "args": args,
                        "train_steps": train_steps
                    }"""

content = content.replace(old, new)

# Fix 5 — add time limit to training
old = """            train_steps += 1"""

new = """            train_steps += 1
            if args.time_limit_hours is not None:
              elapsed = time() - run_start_time
              limit = args.time_limit_hours * 3600
              buffer = args.time_buffer_minutes * 60

              if elapsed >= limit - buffer:
                  if accelerator.is_main_process:
                      checkpoint = {
                          "model": accelerator.unwrap_model(model).state_dict(),
                          "ema": ema.state_dict(),
                          "opt": opt.state_dict(),
                          "args": args,
                          "train_steps": train_steps,
                          "epoch": epoch,
                      }
                      checkpoint_path = f"{checkpoint_dir}/{train_steps:07d}_time_limit.pt"
                      torch.save(checkpoint, checkpoint_path)
                      logger.info(f"Time limit approaching. Saved checkpoint to {checkpoint_path}")
                  return
"""

content = content.replace(old, new)

# Fix 6 - Fix logger configuration for Colab/Jupyter
# Adds force=True so logging.basicConfig actually installs the StreamHandler + FileHandler.

old = """    logging.basicConfig(
        level=logging.INFO,
        format='[\\033[34m%(asctime)s\\033[0m] %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[logging.StreamHandler(), logging.FileHandler(f"{logging_dir}/log.txt")]
    )"""

new = """    logging.basicConfig(
        level=logging.INFO,
        format='[\\033[34m%(asctime)s\\033[0m] %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[logging.StreamHandler(), logging.FileHandler(f"{logging_dir}/log.txt")],
        force=True
    )"""

content = content.replace(old, new)

# Fix 7 — argparse: add new argparse arguments
old = """    parser.add_argument("--num-classes", type=int, default=1000)"""

new = """    parser.add_argument("--num-classes", type=int, default=1102)
    parser.add_argument("--pretrained", type=str, default=None)
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--time-limit-hours", type=float, default=None)
    parser.add_argument("--time-buffer-minutes", type=float, default=30)"""

content = content.replace(old, new)

with open('/content/fast-DiT/train.py', 'w') as f:
    f.write(content)

print("All fixes applied")

All fixes applied


In [ ]:
# SMOKE SCREEN TEST - run the training code
# !accelerate launch \
#   --num_processes=1 \
#   --mixed_precision fp16 /content/fast-DiT/train.py \
#   --model DiT-XL/2 \
#   --feature-path /content/features \
#   --pretrained /content/DiT-XL-2-256x256.pt \
#   --results-dir /content/drive/MyDrive/ImageNet_Project/results \
#   --num-classes 1102 \
#   --epochs 1 \
#   --global-batch-size 64 \
#   --ckpt-every 50 \
#   --log-every 5

In [ ]:
# run the training code
!accelerate launch \
  --num_processes=1 \
  --mixed_precision fp16 /content/fast-DiT/train.py \
  --model DiT-XL/2 \
  --feature-path /content/features \
  --pretrained /content/DiT-XL-2-256x256.pt \
  --results-dir /content/drive/MyDrive/ImageNet_Project/results \
  --num-classes 1102 \
  --epochs 50 \
  --global-batch-size 256 \
  --ckpt-every 5000 \
  --log-every 100

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[2026-05-07 08:52:08] Experiment directory created at /content/drive/MyDrive/ImageNet_Project/results/007-DiT-XL-2
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
[2026-05-07 08:52:18] HTTP Request: H

In [ ]:
# Wait and verify that google drive has synced before deleting runtime
!ls -lh /content/drive/MyDrive/ImageNet_Project/results/007-DiT-XL-2/checkpoints

# also force a sync
!sync

total 41G
-rw------- 1 root root 11G May  7 11:57 0005000.pt
-rw------- 1 root root 11G May  7 15:03 0010000.pt
-rw------- 1 root root 11G May  7 18:10 0015000.pt
-rw------- 1 root root 11G May  7 21:17 0020000.pt


In [ ]:
# save training script once patches have been applied
# from google.colab import files
# import os

# train_path = "/content/fast-DiT/train.py"

# assert os.path.exists(train_path), f"File not found: {train_path}"

# files.download(train_path)

In [ ]:
# resets to original training script
# !cd /content/fast-DiT && git checkout train.py

In [ ]:
# function to keep Colab browswer active to prevent session crashes (run in browser console)
# function KeepAlive() {
#   document.querySelector('#top-toolbar').click();
#   console.log('Keeping alive:', new Date());
# }
# setInterval(KeepAlive, 60000);